# Store 1 to Store 2 Item Matching using Word Embeddings
Match clothing items from store_1 to store_2 based on item descriptions using spaCy word embeddings.

In [29]:
import pandas as pd
import numpy as np
from glob import glob
from sklearn.metrics.pairwise import cosine_similarity
import spacy

In [30]:
# Load data efficiently
data_dir = "datasets/"

# Load store_1 files in one operation
store_1_files = sorted(glob(f"{data_dir}store_1*.csv"))
df_1 = pd.concat([pd.read_csv(f) for f in store_1_files], ignore_index=True)

# Load store_2
df_2 = pd.read_csv(f"{data_dir}store_2.csv", low_memory=False)

print(f"Store 1: {df_1.shape}")
print(f"Store 2: {df_2.shape}")

Store 1: (132561, 31)
Store 2: (75634, 27)


In [51]:
# Load spaCy model with word vectors
nlp_model = spacy.load("en_core_web_sm")
spacy_default_vector_dim = 96

def compute_doc_embeddings(series, batch_size=1000):
    """Compute average word embeddings for each document."""
    results = []
    docs = nlp_model.pipe(series.fillna(""), batch_size=batch_size)
    
    for doc in docs:
        # Get vectors for non-stopword, non-punctuation tokens
        vectors = [w.vector for w in doc 
                  if w.has_vector and not (w.is_stop or w.is_space or w.is_punct)]
        
        if len(vectors)>0:
            # Average the word vectors
            doc_vector = np.mean(vectors, axis=0).reshape((1, spacy_default_vector_dim))
        else:
            # Fallback to zero vector if no valid words
            doc_vector = np.zeros((1, spacy_default_vector_dim))

        results.append(doc_vector)
    
    return np.vstack(results)

# Compute embeddings
print("Computing embeddings for store 1...")
X_store1 = compute_doc_embeddings(df_1["Item"])
print(f"Store 1 embeddings shape: {X_store1.shape}")

print("Computing embeddings for store 2...")
X_store2 = compute_doc_embeddings(df_2["Item"])
print(f"Store 2 embeddings shape: {X_store2.shape}")

Computing embeddings for store 1...
Store 1 embeddings shape: (132561, 96)
Computing embeddings for store 2...
Store 2 embeddings shape: (75634, 96)


In [53]:
# Compute similarities in batches to avoid memory issues
batch_size = 5000
matches = []

for start_idx in range(0, len(df_1), batch_size):
    end_idx = start_idx + batch_size
    
    # Compute similarity for this batch
    batch_similarities = cosine_similarity(X_store1[start_idx:end_idx], X_store2)
    
    # Find best match for each item
    best_match_indices = batch_similarities.argmax(axis=1)
    best_match_scores = batch_similarities[range(len(best_match_indices)), best_match_indices]
    
    # Store results
    for i, (match_idx, score) in enumerate(zip(best_match_indices, best_match_scores)):
        matches.append({
            "store_1_index": start_idx + i,
            "store_2_index": match_idx,
            "similarity_score": score
        })
    
    print(f"Processed {end_idx}/{len(df_1)} items")

# Create matches dataframe
matches_df = pd.DataFrame(matches)
matches_df["store_1_item"] = df_1["Item"].values
matches_df["store_2_item"] = df_2["Item"].iloc[matches_df["store_2_index"]].values

Processed 5000/132561 items
Processed 10000/132561 items
Processed 15000/132561 items
Processed 20000/132561 items
Processed 25000/132561 items
Processed 30000/132561 items
Processed 35000/132561 items
Processed 40000/132561 items
Processed 45000/132561 items
Processed 50000/132561 items
Processed 55000/132561 items
Processed 60000/132561 items
Processed 65000/132561 items
Processed 70000/132561 items
Processed 75000/132561 items
Processed 80000/132561 items
Processed 85000/132561 items
Processed 90000/132561 items
Processed 95000/132561 items
Processed 100000/132561 items
Processed 105000/132561 items
Processed 110000/132561 items
Processed 115000/132561 items
Processed 120000/132561 items
Processed 125000/132561 items
Processed 130000/132561 items
Processed 135000/132561 items


In [54]:
# Show match quality statistics
print("Match Quality Statistics:")
print(f"Mean similarity: {matches_df['similarity_score'].mean():.3f}")
print(f"Median similarity: {matches_df['similarity_score'].median():.3f}")
print(f"Min similarity: {matches_df['similarity_score'].min():.3f}")
print(f"Max similarity: {matches_df['similarity_score'].max():.3f}")
print(f"\nScore distribution:")
print(matches_df['similarity_score'].describe())

Match Quality Statistics:
Mean similarity: 0.896
Median similarity: 0.900
Min similarity: 0.000
Max similarity: 1.000

Score distribution:
count    132561.000000
mean          0.895764
std           0.033684
min           0.000000
25%           0.877596
50%           0.899715
75%           0.918669
max           1.000000
Name: similarity_score, dtype: float64


In [55]:
# Show examples of matches at different quality levels
print("\n=== TOP 10 BEST MATCHES ===")
top_matches = matches_df.nlargest(10, 'similarity_score')
for _, row in top_matches.iterrows():
    print(f"\nScore: {row['similarity_score']:.3f}")
    print(f"  Store 1: {row['store_1_item']}")
    print(f"  Store 2: {row['store_2_item']}")

print("\n\n=== 10 MEDIAN MATCHES ===")
median_score = matches_df['similarity_score'].median()
median_matches = matches_df.iloc[(matches_df['similarity_score'] - median_score).abs().argsort()[:10]]
for _, row in median_matches.iterrows():
    print(f"\nScore: {row['similarity_score']:.3f}")
    print(f"  Store 1: {row['store_1_item']}")
    print(f"  Store 2: {row['store_2_item']}")

print("\n\n=== 10 WORST MATCHES ===")
worst_matches = matches_df.nsmallest(10, 'similarity_score')
for _, row in worst_matches.iterrows():
    print(f"\nScore: {row['similarity_score']:.3f}")
    print(f"  Store 1: {row['store_1_item']}")
    print(f"  Store 2: {row['store_2_item']}")


=== TOP 10 BEST MATCHES ===

Score: 1.000
  Store 1: Steel Spikes Pyramid 6mm
  Store 2: Steel Spikes Pyramid 6mm

Score: 1.000
  Store 1: Shopify Refund
  Store 2: Shopify Refund

Score: 1.000
  Store 1: Salomon Soft Reservoir 1.5L
  Store 2: Salomon Soft Reservoir 1.5L

Score: 1.000
  Store 1: Salomon Soft Reservoir 1.5L
  Store 2: Salomon Soft Reservoir 1.5L

Score: 1.000
  Store 1: Shopify Discount
  Store 2: Shopify Discount

Score: 1.000
  Store 1: Salomon Soft Reservoir 1.5L
  Store 2: Salomon Soft Reservoir 1.5L

Score: 1.000
  Store 1: Hydrapak Bite Valve Sheath
  Store 2: Hydrapak Bite Valve Sheath

Score: 1.000
  Store 1: No product found
  Store 2: No product found

Score: 1.000
  Store 1: Shopify Discount
  Store 2: Shopify Discount

Score: 1.000
  Store 1: Salomon Soft Reservoir 1.5L
  Store 2: Salomon Soft Reservoir 1.5L


=== 10 MEDIAN MATCHES ===

Score: 0.900
  Store 1: M New Balance MTHIERD5 2E 6
  Store 2: NB FuelCell Rebel 2E 12

Score: 0.900
  Store 1: W New Bala

In [56]:
# Show 20 random examples
print("\n\n=== 20 RANDOM MATCH EXAMPLES ===")
random_matches = matches_df.sample(20, random_state=42)
for _, row in random_matches.iterrows():
    print(f"\nScore: {row['similarity_score']:.3f}")
    print(f"  Store 1: {row['store_1_item']}")
    print(f"  Store 2: {row['store_2_item']}")



=== 20 RANDOM MATCH EXAMPLES ===

Score: 0.809
  Store 1: W New Balance W880HP8 2A 7
  Store 2: W NB W860V9 2A 7

Score: 0.832
  Store 1: W New Balance WW813GY1 2A 10
  Store 2: W Brooks Adrenaline GTS 20 2A 11

Score: 0.871
  Store 1: M New Balance M880A10 2E 11.5
  Store 2: W Saucony Guide ISO 2 D 9.5

Score: 0.923
  Store 1: W New Balance 880v13 W880B13 2E 10
  Store 2: W Nike Air Zoom Pegasus 36 B 6.5

Score: 0.888
  Store 1: K New Balance YP680BB6 W 6.5
  Store 2: Garmin Instinct E Electric Lime 45mm

Score: 0.926
  Store 1: Asics Thermal Run Glove - S-44197 0090 S/M
  Store 2: Buff Sun Bucket Hat (Ocher) S/M

Score: 0.905
  Store 1: M New Balance MW577WT 2E 14
  Store 2: W Smartwool Merino Sport 250 LS 1/4 Zip All Col MD

Score: 0.904
  Store 1: M New Balance M1080K13 2E 12
  Store 2: W Saucony Peregrine 10 ST B 6.5

Score: 0.923
  Store 1: Run Cold Weather TC Crew Socks Alpine Blue MD
  Store 2: New Balance Track To Trail Crew Sweatshirt Grey MD

Score: 0.833
  Store 1: Yak Tr

In [ ]:
import matplotlib.pyplot as plt

plt.plot(batch_similarities[0])
plt.show()